# ReconBot Dense Reconstruction ? Colab T4 (Rebuilt)

This notebook completes the current `hallway_room_001` experiment using the two validated sparse components. It produces:

- CUDA PatchMatch dense clouds for both components,
- one metric fused point cloud,
- PLY, OBJ, and GLB meshes,
- a quantitative summary in Google Drive.

## Why this notebook is different

This version does **not** install COLMAP dependencies with `apt`. Colab's system OpenImageIO/OpenEXR packages can conflict. Instead, all C++ dependencies live in an isolated conda-forge environment created by micromamba. The system is used only for the existing NVIDIA driver, CUDA toolkit, GCC, and basic shell utilities.

It also avoids Git short-SHA fetching by downloading a source archive directly, prints complete build errors, saves logs to Drive, and caches completed component clouds.

## Before running

1. In Google Drive create `My Drive/ReconBotDense`.
2. Upload `reconbot_dense_colab_input.zip` into that folder.
3. In Colab select **Runtime ? Change runtime type ? T4 GPU**.
4. Run the cells in order.

The default is a fast 720-pixel dense pass. It is designed to finish first; quality can be increased later.


In [ ]:
# Cell 1 ? imports and robust command helpers
from __future__ import annotations

import hashlib
import json
import os
import platform
import shutil
import subprocess
import tarfile
import time
import urllib.request
import zipfile
from pathlib import Path

def run(command, *, cwd=None, env=None, log_path=None, check=True):
    command = [str(item) for item in command]
    print('+', ' '.join(command), flush=True)
    if log_path is None:
        return subprocess.run(command, cwd=cwd, env=env, check=check, text=True)
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.run(
            command, cwd=cwd, env=env, stdout=log, stderr=subprocess.STDOUT, text=True
        )
    if process.returncode != 0:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print('\n'.join(lines[-120:]))
        raise RuntimeError(
            f'Command failed with exit code {process.returncode}. Full log: {log_path}'
        )
    return process

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


In [ ]:
# Cell 2 ? GPU and runtime preflight
print('Python:', platform.python_version())
print(Path('/etc/os-release').read_text())
run(['nvidia-smi'])
run(['/usr/local/cuda/bin/nvcc', '--version'])

gpu_name = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True
).strip()
print('Assigned GPU:', gpu_name)
assert gpu_name, 'No NVIDIA GPU detected. Change the Colab runtime to T4 GPU.'


In [ ]:
# Cell 3 ? mount Drive and configure the fast first pass
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/ReconBotDense')
INPUT_ZIP = DRIVE_ROOT / 'reconbot_dense_colab_input.zip'
RESULTS_DIR = DRIVE_ROOT / 'results'
CACHE_DIR = DRIVE_ROOT / 'cache'
BUILD_LOG_DIR = DRIVE_ROOT / 'build_logs'
COMPONENT_CACHE = DRIVE_ROOT / 'component_cache'
WORK_ROOT = Path('/content/reconbot_dense')

# Fastest practical first pass. Increase to 960 and 3 iterations only after success.
MAX_IMAGE_SIZE = 720
PATCHMATCH_ITERATIONS = 2
PATCHMATCH_SAMPLES = 8
VOXEL_SIZE_M = 0.02
POISSON_DEPTH = 8

COLMAP_COMMIT = '5b76f53'
EXPECTED_INPUT_SHA256 = 'ece58e69d7f98b7d51c1b99a3b6b193aabe1f85cc22a6879f94d456eca791e71'

for directory in (DRIVE_ROOT, RESULTS_DIR, CACHE_DIR, BUILD_LOG_DIR, COMPONENT_CACHE, WORK_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

assert INPUT_ZIP.is_file(), f'Missing upload: {INPUT_ZIP}'
actual_hash = sha256(INPUT_ZIP)
print('Input ZIP:', INPUT_ZIP)
print('Size:', round(INPUT_ZIP.stat().st_size / 1024**2, 1), 'MB')
print('SHA-256:', actual_hash)
assert actual_hash == EXPECTED_INPUT_SHA256, 'Input ZIP is incomplete or is the wrong version.'


## Isolated dependency environment

This cell installs no Ubuntu development packages. It creates `/content/colmap-env` from conda-forge, ensuring OpenImageIO, OpenEXR, Imath, Ceres, SuiteSparse, and related libraries are mutually compatible.


In [ ]:
# Cell 4 ? install micromamba and create an isolated COLMAP dependency environment
MICROMAMBA_ROOT = Path('/content/micromamba-root')
MICROMAMBA = Path('/content/bin/micromamba')
ENV_PREFIX = Path('/content/colmap-env')
MAMBA_ARCHIVE = Path('/content/micromamba.tar.bz2')

if not MICROMAMBA.is_file():
    print('Downloading micromamba...')
    urllib.request.urlretrieve(
        'https://micro.mamba.pm/api/micromamba/linux-64/latest', MAMBA_ARCHIVE
    )
    run(['tar', '-xjf', MAMBA_ARCHIVE, '-C', '/content', 'bin/micromamba'])

mamba_env = os.environ.copy()
mamba_env['MAMBA_ROOT_PREFIX'] = str(MICROMAMBA_ROOT)

packages = [
    'python=3.11',
    'cmake>=3.28',
    'ninja',
    'ccache',
    'pkg-config',
    'boost',
    'eigen=3.4.0',
    'openimageio',
    'openexr',
    'imath',
    'curl',
    'metis',
    'glog',
    'gflags',
    'gtest',
    'ceres-solver=2.2.0=cpugplhc142d66_210',
    'suitesparse',
    'glew',
    'sqlite',
    'cgal-cpp',
    'openblas',
    'libblas=*=*openblas',
]

if not (ENV_PREFIX / 'bin/cmake').is_file():
    run(
        [MICROMAMBA, 'create', '-y', '-p', ENV_PREFIX, '-c', 'conda-forge',
         '--strict-channel-priority', *packages],
        env=mamba_env,
        log_path=BUILD_LOG_DIR / 'micromamba_create.log',
    )
else:
    print('Dependency environment already exists:', ENV_PREFIX)

# Repair existing environments that may contain a CUDA 12.9 Ceres build.
# Ceres runs bundle adjustment on CPU; COLMAP PatchMatch still uses the T4 GPU.
run(
    [MICROMAMBA, 'install', '-y', '-p', ENV_PREFIX,
     '-c', 'conda-forge', '--strict-channel-priority',
     'ceres-solver=2.2.0=cpugplhc142d66_210'],
    env=mamba_env,
    log_path=BUILD_LOG_DIR / 'micromamba_cpu_ceres.log',
)

run([ENV_PREFIX / 'bin/cmake', '--version'])
run([ENV_PREFIX / 'bin/ninja', '--version'])
oiio_configs = list(ENV_PREFIX.rglob('OpenImageIOConfig.cmake'))
openexr_configs = list(ENV_PREFIX.rglob('OpenEXRConfig.cmake'))
assert oiio_configs, 'OpenImageIO CMake configuration was not installed.'
assert openexr_configs, 'OpenEXR CMake configuration was not installed.'
print('OpenImageIO config:', oiio_configs[0])
print('OpenEXR config:', openexr_configs[0])


## Build CUDA COLMAP

The exact source archive is downloaded directly?there is no Git fetch or checkout. CMake output and compiler output are written to `ReconBotDense/build_logs` in Drive. If a command fails, the final 120 log lines are printed automatically.

The first successful build is cached in Drive. `ccache` also survives runtime interruptions.


In [ ]:
# Cell 5 ? configure and build CUDA COLMAP using the isolated dependencies
# COLMAP links OpenGL even in a headless GUI-disabled build.
# Install only the Mesa development interface required by CMake.
run(['apt-get', 'update', '-qq'],
    log_path=BUILD_LOG_DIR / 'apt_opengl_update.log')
run(['apt-get', 'install', '-y', '-qq', 'libgl1-mesa-dev'],
    log_path=BUILD_LOG_DIR / 'apt_opengl_install.log')

SOURCE = Path('/content/colmap-src')
BUILD = Path('/content/colmap-build')
COLMAP_PREFIX = Path('/content/colmap-install')
COLMAP_CACHE = CACHE_DIR / f'colmap_cuda_{COLMAP_COMMIT}_native.tar.gz'
SOURCE_ARCHIVE = Path('/content/colmap-source.tar.gz')
CCACHE_DIR = CACHE_DIR / 'ccache'

BUILD_ENV = os.environ.copy()
BUILD_ENV.update({
    'PATH': f'{ENV_PREFIX}/bin:/usr/local/cuda/bin:' + BUILD_ENV.get('PATH', ''),
    'LD_LIBRARY_PATH': (
        f'{ENV_PREFIX}/lib:/usr/local/cuda/lib64:' + BUILD_ENV.get('LD_LIBRARY_PATH', '')
    ),
    # Make GCC link against conda's matching C++ runtime, not Colab's older system copy.
    'LIBRARY_PATH': f'{ENV_PREFIX}/lib:' + BUILD_ENV.get('LIBRARY_PATH', ''),
    'CMAKE_PREFIX_PATH': str(ENV_PREFIX),
    'PKG_CONFIG_PATH': f'{ENV_PREFIX}/lib/pkgconfig:{ENV_PREFIX}/share/pkgconfig',
    'CC': '/usr/bin/gcc',
    'CXX': '/usr/bin/g++',
    'CUDAHOSTCXX': '/usr/bin/g++',
    'CCACHE_DIR': str(CCACHE_DIR),
    'CCACHE_MAXSIZE': '8G',
})
CCACHE_DIR.mkdir(parents=True, exist_ok=True)

shutil.rmtree(COLMAP_PREFIX, ignore_errors=True)
if COLMAP_CACHE.is_file():
    print('Restoring cached CUDA COLMAP:', COLMAP_CACHE)
    with tarfile.open(COLMAP_CACHE, 'r:gz') as archive:
        archive.extractall('/content')
else:
    shutil.rmtree(SOURCE, ignore_errors=True)
    shutil.rmtree(BUILD, ignore_errors=True)
    SOURCE.mkdir(parents=True)
    BUILD.mkdir(parents=True)

    source_url = f'https://github.com/colmap/colmap/archive/{COLMAP_COMMIT}.tar.gz'
    print('Downloading exact COLMAP source archive:', source_url)
    urllib.request.urlretrieve(source_url, SOURCE_ARCHIVE)
    run(['tar', '-xzf', SOURCE_ARCHIVE, '--strip-components=1', '-C', SOURCE])

    cmake_command = [
        ENV_PREFIX / 'bin/cmake',
        '-S', SOURCE,
        '-B', BUILD,
        '-GNinja',
        '-DCMAKE_BUILD_TYPE=Release',
        '-DGUI_ENABLED=OFF',
        '-DTESTS_ENABLED=OFF',
        '-DCUDA_ENABLED=ON',
        '-DCMAKE_CUDA_ARCHITECTURES=native',
        '-DCMAKE_CUDA_COMPILER=/usr/local/cuda/bin/nvcc',
        '-DCMAKE_CUDA_HOST_COMPILER=/usr/bin/g++',
        '-DCMAKE_C_COMPILER=/usr/bin/gcc',
        '-DCMAKE_CXX_COMPILER=/usr/bin/g++',
        f'-DCMAKE_PREFIX_PATH={ENV_PREFIX}',
        f'-DCMAKE_INSTALL_PREFIX={COLMAP_PREFIX}',
        f'-DCMAKE_INSTALL_RPATH={ENV_PREFIX}/lib',
        '-DCMAKE_BUILD_WITH_INSTALL_RPATH=ON',
        '-DCMAKE_C_COMPILER_LAUNCHER=ccache',
        '-DCMAKE_CXX_COMPILER_LAUNCHER=ccache',
        '-DCMAKE_CUDA_COMPILER_LAUNCHER=ccache',
        '-DBLA_VENDOR=OpenBLAS',
        '-DINTERPROCEDURAL_OPTIMIZATION_ENABLED=OFF',
    ]
    run(
        cmake_command, env=BUILD_ENV,
        log_path=BUILD_LOG_DIR / 'colmap_cmake_configure.log'
    )
    run(
        [ENV_PREFIX / 'bin/ninja', '-C', BUILD, '-j2'],
        env=BUILD_ENV, log_path=BUILD_LOG_DIR / 'colmap_ninja_build.log'
    )
    run(
        [ENV_PREFIX / 'bin/ninja', '-C', BUILD, 'install'],
        env=BUILD_ENV, log_path=BUILD_LOG_DIR / 'colmap_ninja_install.log'
    )
    with tarfile.open(COLMAP_CACHE, 'w:gz') as archive:
        archive.add(COLMAP_PREFIX, arcname='colmap-install')
    print('Cached CUDA COLMAP:', COLMAP_CACHE)

COLMAP = COLMAP_PREFIX / 'bin/colmap'
assert COLMAP.is_file(), f'COLMAP executable not found: {COLMAP}'
run([COLMAP, '-h'], env=BUILD_ENV, log_path=BUILD_LOG_DIR / 'colmap_help.log')
help_text = (BUILD_LOG_DIR / 'colmap_help.log').read_text(errors='replace')
print('\n'.join(help_text.splitlines()[:8]))
assert 'CUDA' in help_text and 'without CUDA' not in help_text, 'COLMAP was not built with CUDA.'
print('CUDA COLMAP is ready:', COLMAP)


In [ ]:
# Cell 6 ? extract and verify the two registered components
INPUT_ROOT = WORK_ROOT / 'input'
shutil.rmtree(INPUT_ROOT, ignore_errors=True)
INPUT_ROOT.mkdir(parents=True)
with zipfile.ZipFile(INPUT_ZIP) as archive:
    assert archive.testzip() is None, 'The input ZIP is damaged.'
    archive.extractall(INPUT_ROOT)
manifest = json.loads((INPUT_ROOT / 'manifest.json').read_text())

for name, metadata in manifest['components'].items():
    images = list((INPUT_ROOT / name / 'images').glob('*.jpg'))
    sparse = INPUT_ROOT / name / 'sparse'
    assert len(images) == metadata['registered_images'], (name, len(images), metadata)
    for filename in ('cameras.txt', 'images.txt', 'points3D.txt'):
        assert (sparse / filename).is_file(), sparse / filename

print(json.dumps({
    'dataset_id': manifest['dataset_id'],
    'components': manifest['components'],
    'stitch_confidence': manifest['stitch_confidence'],
    'seam_geometry_gate_passed': manifest['seam_geometry_gate_passed'],
}, indent=2))


## CUDA dense reconstruction

Each component is cached to Drive immediately after fusion. If Colab disconnects after one component completes, rerun the notebook: the completed cloud will be restored instead of reconstructed.

At the default 720-pixel profile, a T4 typically takes tens of minutes rather than hours. Actual Colab load varies.


In [ ]:
# Cell 7 ? reusable dense reconstruction function with resume support
def colmap_run(arguments, log_name):
    return run(
        [COLMAP, *arguments], env=BUILD_ENV,
        log_path=BUILD_LOG_DIR / log_name
    )

def dense_component(name):
    cached_fused = COMPONENT_CACHE / name / (
        f'fused_{MAX_IMAGE_SIZE}px_{PATCHMATCH_ITERATIONS}iter.ply'
    )
    cached_fused.parent.mkdir(parents=True, exist_ok=True)
    if cached_fused.is_file() and cached_fused.stat().st_size > 1024:
        print('Using cached dense cloud:', cached_fused)
        return cached_fused

    component = INPUT_ROOT / name
    workspace = WORK_ROOT / name / 'dense'
    shutil.rmtree(workspace.parent, ignore_errors=True)
    workspace.parent.mkdir(parents=True)
    started = time.time()

    colmap_run([
        'image_undistorter',
        '--image_path', component / 'images',
        '--input_path', component / 'sparse',
        '--output_path', workspace,
        '--output_type', 'COLMAP',
        '--max_image_size', str(MAX_IMAGE_SIZE),
    ], f'{name}_undistorter.log')

    colmap_run([
        'patch_match_stereo',
        '--workspace_path', workspace,
        '--workspace_format', 'COLMAP',
        '--PatchMatchStereo.gpu_index', '0',
        '--PatchMatchStereo.geom_consistency', '1',
        '--PatchMatchStereo.num_iterations', str(PATCHMATCH_ITERATIONS),
        '--PatchMatchStereo.num_samples', str(PATCHMATCH_SAMPLES),
        '--PatchMatchStereo.window_radius', '3',
        '--PatchMatchStereo.cache_size', '2',
    ], f'{name}_patch_match.log')

    local_fused = workspace / 'fused.ply'
    colmap_run([
        'stereo_fusion',
        '--workspace_path', workspace,
        '--workspace_format', 'COLMAP',
        '--input_type', 'geometric',
        '--output_path', local_fused,
    ], f'{name}_stereo_fusion.log')

    shutil.copy2(local_fused, cached_fused)
    print(name, 'finished in', round((time.time() - started) / 60, 1), 'minutes')
    print('Cached:', cached_fused, round(cached_fused.stat().st_size / 1024**2, 1), 'MB')
    return cached_fused


In [ ]:
# Cell 8 ? reconstruct the smaller secondary component first
secondary_fused = dense_component('secondary_component')


In [ ]:
# Cell 9 ? reconstruct the main component
main_fused = dense_component('main_component')


## Metric fusion and mesh export

The saved transforms place both dense clouds into meters. Poisson meshing is performed separately for each component before combining them, preventing the mesher from inventing a solid bridge across the unsupported seam.


In [ ]:
# Cell 10 ? clean, transform, fuse, and mesh
run([os.sys.executable, '-m', 'pip', 'install', '-q', 'open3d', 'trimesh'])
import numpy as np
import open3d as o3d
import trimesh

main_transform = np.asarray(manifest['main_to_metric_transform'], dtype=float)
secondary_transform = np.asarray(manifest['secondary_to_metric_transform'], dtype=float)

def load_metric_cloud(path, transform):
    cloud = o3d.io.read_point_cloud(str(path))
    assert len(cloud.points) > 0, f'Empty dense cloud: {path}'
    cloud.transform(transform)
    return cloud

def clean_cloud(cloud):
    cloud = cloud.voxel_down_sample(VOXEL_SIZE_M)
    cloud, _ = cloud.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.5)
    cloud.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=0.10, max_nn=40)
    )
    return cloud

def poisson_component(cloud):
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        cloud, depth=POISSON_DEPTH, linear_fit=True
    )
    densities = np.asarray(densities)
    mesh.remove_vertices_by_mask(densities < np.quantile(densities, 0.03))
    mesh = mesh.crop(cloud.get_axis_aligned_bounding_box())
    mesh.compute_vertex_normals()
    return mesh

main_cloud = clean_cloud(load_metric_cloud(main_fused, main_transform))
secondary_cloud = clean_cloud(load_metric_cloud(secondary_fused, secondary_transform))
combined_cloud = (main_cloud + secondary_cloud).voxel_down_sample(VOXEL_SIZE_M)

cloud_path = RESULTS_DIR / 'hallway_room_001_dense_metric.ply'
o3d.io.write_point_cloud(str(cloud_path), combined_cloud, write_ascii=False)

main_mesh = poisson_component(main_cloud)
secondary_mesh = poisson_component(secondary_cloud)
combined_mesh = main_mesh + secondary_mesh
mesh_ply = RESULTS_DIR / 'hallway_room_001_mesh_metric.ply'
mesh_obj = RESULTS_DIR / 'hallway_room_001_mesh_metric.obj'
mesh_glb = RESULTS_DIR / 'hallway_room_001_mesh_metric.glb'
o3d.io.write_triangle_mesh(str(mesh_ply), combined_mesh, write_ascii=False)
o3d.io.write_triangle_mesh(str(mesh_obj), combined_mesh, write_ascii=False)
trimesh.load_mesh(mesh_ply, process=False).export(mesh_glb)

summary = {
    'dense_points': len(combined_cloud.points),
    'mesh_vertices': len(combined_mesh.vertices),
    'mesh_triangles': len(combined_mesh.triangles),
    'max_image_size': MAX_IMAGE_SIZE,
    'patchmatch_iterations': PATCHMATCH_ITERATIONS,
    'voxel_size_m': VOXEL_SIZE_M,
    'poisson_depth': POISSON_DEPTH,
    'stitch_confidence': manifest['stitch_confidence'],
    'seam_geometry_gate_passed': manifest['seam_geometry_gate_passed'],
    'outputs': [str(cloud_path), str(mesh_ply), str(mesh_obj), str(mesh_glb)],
}
(RESULTS_DIR / 'dense_reconstruction_summary.json').write_text(
    json.dumps(summary, indent=2) + '\n'
)
print(json.dumps(summary, indent=2))


In [ ]:
# Cell 11 ? final deliverables
print('Results saved to:', RESULTS_DIR)
for path in sorted(RESULTS_DIR.iterdir()):
    print(f'{path.name:48s} {path.stat().st_size / 1024**2:9.1f} MB')
print('\nOpen hallway_room_001_mesh_metric.glb in Blender or a glTF viewer.')


## Recovery guide

- **Micromamba failure:** open `ReconBotDense/build_logs/micromamba_create.log`. Restart the runtime only if the download was interrupted.
- **CMake failure:** open `colmap_cmake_configure.log`; the notebook prints its final 120 lines automatically. Do not install random `apt` packages.
- **Compiler failure:** open `colmap_ninja_build.log`. Rerunning uses the Drive-backed ccache.
- **CUDA out of memory:** change `MAX_IMAGE_SIZE` from 720 to 540, then rerun from Cell 7. Delete the relevant cached fused PLY first.
- **Colab disconnect:** rerun all cells. Any completed component fused cloud is reused from Drive.
- **Higher quality:** after the fast run succeeds, set `MAX_IMAGE_SIZE=960`, `PATCHMATCH_ITERATIONS=3`, and `PATCHMATCH_SAMPLES=10`.
